In [ ]:
import sys
sys.path.insert(0, "/projappl/project_2012747/mars/MarS")  # folder that contains market_simulation/
from market_simulation.models.order_model import OrderModel

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.6/536.6 kB 11.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 53.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━━ 4/5 [huggingface_hub]  WARNING: The scripts hf and tiny-agents are installed in '/users/pst/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [huggingface_hub] [huggingface_hub]


ModuleNotFoundError: No module named 'huggingface_hub'

In [4]:
import os


sweep_configuration = {
    "method": "grid",
    "metric": {"goal": "minimize", "name": "val/loss"},
    "parameters": {
        "train_fraction": {"values": [1.0, 0.5]},
        "model_variant": {"values": ["base", "small"]},

        # keep a few training knobs configurable if you want
        "lr": {"value": 3e-4},
        "batch_size": {"value": 8},
        "max_steps": {"value": 20000},
        "eval_every": {"value": 100},
        "val_max_batches": {"value": 200},
        "seed": {"value": 123},
    },
}


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /users/pst/.netrc


Create sweep with ID: o8kqccu1
Sweep URL: https://wandb.ai/aymeric-b/MarS/sweeps/o8kqccu1
SWEEP_ID: o8kqccu1


In [ ]:
import glob, bisect
import os
import numpy as np
import zarr, torch
import torch.nn.functional as F
from tqdm import tqdm
from torch.utils.data import Dataset, IterableDataset, DataLoader
from datasets import load_dataset
from zarr.storage import ZipStore
from functools import lru_cache

from market_simulation.models.order_model import OrderModel





# -------------------------
# Fixed config (shared)
# -------------------------
K = 1024
batch_size = 4096
train_fraction=0.8
seed=42
TRAIN_STRIDE = 16
VAL_STRIDE = 16

USE_AMP = True
AMP_DTYPE = torch.bfloat16  # or torch.float16

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

feat_cols = [f"f{i}" for i in range(15)]


def add_f4(batch):
    t = np.asarray(batch["Time"], dtype=np.int64)
    t_sec = (t // 1_000_000_000).astype(np.int64)
    batch["f4"] = np.clip(t_sec - 34200, 0, 23399).astype(np.int64)
    return batch


def load_parquet_as_cols(path: str):
    ds = load_dataset("parquet", data_files={"data": path})["data"]
    ds = ds.map(add_f4, batched=True, batch_size=200_000, num_proc=1)

    # keep only f0..f14
    drop_cols = [c for c in ds.column_names if c not in feat_cols]
    if drop_cols:
        ds = ds.remove_columns(drop_cols)

    ds = ds.with_format("numpy")
    cols = {c: ds[c] for c in feat_cols}
    return cols





class MultiZarrOrderDataset(Dataset):
    def __init__(self, train_dir="train", pattern="*_features.zarr.zip", seq_len=1024):
        self.seq_len = seq_len
        self.paths = sorted(glob.glob(f"{train_dir}/{pattern}"))

        # length per file = N - seq_len (each idx predicts next row)
        self.lens = []
        for p in self.paths:
            with ZipStore(p, mode="r") as store:
                X = zarr.open(store=store, path="X", mode="r")
                self.lens.append(X.shape[0] - seq_len)

        self.cum = []
        s = 0
        for L in self.lens:
            s += max(0, L)
            self.cum.append(s)

    def __len__(self):
        return self.cum[-1] if self.cum else 0

    @staticmethod
    @lru_cache(maxsize=8)  # per-worker cache; keeps a few files open
    def _open_X(path):
        store = ZipStore(path, mode="r")
        root = zarr.open(store=store, mode="r")
        return store, root["X"]  # keep store alive

    def __getitem__(self, idx):
        fi = bisect.bisect_right(self.cum, idx)
        prev = 0 if fi == 0 else self.cum[fi - 1]
        j = idx - prev

        store, X = self._open_X(self.paths[fi])

        x = X[j : j + self.seq_len]          # (1024, 15) lazy slice
        y = X[j + self.seq_len, 0]           # next order index (f0)

        return torch.from_numpy(x).long(), torch.tensor(y, dtype=torch.long)




def lm_loss_all_positions(logits: torch.Tensor, X: torch.Tensor) -> torch.Tensor:
    targets = X[:, :, 0]           # (B, K)
    logits_s = logits[:, :-1, :]   # (B, K-1, vocab)
    targ_s   = targets[:, 1:]      # (B, K-1)
    return F.cross_entropy(
        logits_s.reshape(-1, logits_s.size(-1)),
        targ_s.reshape(-1),
        reduction="mean",
    )


def pick_train_files(all_train_files, train_fraction: float, seed: int):
    """
    Implements "half the training set" as half the *files* (deterministic shuffle).
    """
    if train_fraction >= 0.999:
        return all_train_files

    if train_fraction <= 0.0:
        raise ValueError("train_fraction must be > 0")

    rng = np.random.default_rng(seed)
    files = list(all_train_files)
    rng.shuffle(files)
    n = max(1, int(round(len(files) * train_fraction)))
    return sorted(files[:n])


def build_model_from_variant(model_variant: str):
    """
    base ~ your current config (emb=64, layers=2, heads=4)
    small = fewer params (emb=48, layers=1, heads=4) -> significantly smaller
    """
    if model_variant == "base":
        EMB_DIM, NUM_LAYERS, NUM_HEADS = 64, 2, 4
    elif model_variant == "small":
        # smaller than base; keep heads dividing emb_dim nicely
        EMB_DIM, NUM_LAYERS, NUM_HEADS = 48, 1, 4
    else:
        raise ValueError(f"Unknown model_variant={model_variant}")

    model = OrderModel(
        emb_dim=EMB_DIM,
        num_layers=NUM_LAYERS,
        num_heads=NUM_HEADS,
        num_max_orders=K,
    ).to(device)

    return model, {"emb_dim": EMB_DIM, "num_layers": NUM_LAYERS, "num_heads": NUM_HEADS}


@torch.no_grad()
def compute_val_loss(model, val_dl, val_max_batches: int | None):
    model.eval()
    total = 0.0
    count = 0

    for b, X in enumerate(val_dl, start=1):
        X = X.to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda"), dtype=AMP_DTYPE):
            logits = model(X)
            loss = lm_loss_all_positions(logits, X)

        n = X.size(0) * (X.size(1) - 1)
        total += loss.item() * n
        count += n

        if val_max_batches is not None and b >= val_max_batches:
            break

    return total / max(1, count)


def train_fn():



    # reproducibility
    torch.manual_seed(int(seed))
    np.random.seed(int(seed))

    # # -------------------------
    # # Data selection (train_fraction)
    # # -------------------------
    # all_train_files = sorted(glob.glob("../data/features/train_*.parquet"))
    # val_files       = sorted(glob.glob("../data/features/test_*.parquet"))
    # train_files = pick_train_files(all_train_files, float(train_fraction), int(seed))
    # # Load segments
    # train_segments = [load_parquet_as_cols(p) for p in train_files]
    # val_segments   = [load_parquet_as_cols(p) for p in val_files]

    ds = MultiZarrOrderDataset("../data/order_model/train", "*_features.zarr.zip", seq_len=1024)

    dl = DataLoader(
        ds,
        batch_size=4096,
        shuffle=True,
        num_workers=8,
        pin_memory=True,
        drop_last=True,
    )



    print("val_segments")
    exit()

    # DataLoaders
    train_iterable = SegmentedBatchedWindowIterable(
        train_segments, K=K, stride=TRAIN_STRIDE, batch_size=int(batch_size),
        random=True, seed=int(seed)
    )
    val_iterable = SegmentedBatchedWindowIterable(
        val_segments, K=K, stride=VAL_STRIDE, batch_size=int(batch_size),
        random=False, seed=int(seed)
    )

    train_dl = DataLoader(train_iterable, batch_size=None, num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_iterable,   batch_size=None, num_workers=2, pin_memory=True)


    # -------------------------
    # Model selection (model_variant)
    # -------------------------
    model, model_hps = build_model_from_variant(str(model_variant))
    n_params = sum(p.numel() for p in model.parameters())

    opt = torch.optim.AdamW(model.parameters(), lr=float(lr))

    use_scaler = (USE_AMP and device.type == "cuda" and AMP_DTYPE == torch.float16)
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

    # -------------------------
    # Train loop
    # -------------------------
    model.train()
    train_it = iter(train_dl)

    max_steps = int(max_steps)
    eval_every = int(eval_every)
    val_max_batches = int(val_max_batches) if val_max_batches is not None else None

    pbar = tqdm(range(1, max_steps + 1), desc=f"train ({model_variant}, frac={train_fraction})")
    for step in pbar:
        X = next(train_it)
        X = X.to(device, non_blocking=True)

        opt.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(USE_AMP and device.type == "cuda"), dtype=AMP_DTYPE):
            logits = model(X)
            loss = lm_loss_all_positions(logits, X)

        if use_scaler:
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            opt.step()

        pbar.set_postfix(train_loss=f"{loss.item():.4f}", params=f"{n_params/1e6:.2f}M")

        if step % eval_every == 0:
            val_loss = compute_val_loss(model, val_dl, val_max_batches)
            print(f"\nstep {step:6d} | val_loss {val_loss:.4f} | params {n_params/1e6:.2f}M\n")
            model.train()




if __name__ == "__main__":
    train_fn()


wandb: Agent Starting Run: aexqqg8t with config:
wandb: 	batch_size: 8
wandb: 	eval_every: 100
wandb: 	lr: 0.0003
wandb: 	max_steps: 20000
wandb: 	model_variant: base
wandb: 	seed: 123
wandb: 	train_fraction: 1
wandb: 	val_max_batches: 200
wandb: WARNING WANDB_NOTEBOOK_NAME should be a path to a notebook file, couldn't find mars_order_model_sweep.


wandb: 
wandb: 🚀 View run floral-sweep-1 at: https://wandb.ai/aymeric-b/MarS/runs/iua74vjg
wandb: Find logs at: wandb/run-20260113_214120-iua74vjg/logs


train (base, frac=1):   1%|          | 102/20000 [02:01<32:43:30,  5.92s/it, params=10.18M, train_loss=7.6563]


step    100 | val_loss 8.7626 | params 10.18M



train (base, frac=1):   1%|          | 202/20000 [03:59<36:01:17,  6.55s/it, params=10.18M, train_loss=5.8124]

...

train (base, frac=1):   5%|▍         | 902/20000 [18:01<30:52:19,  5.82s/it, params=10.18M, train_loss=3.2706]


step    900 | val_loss 7.6182 | params 10.18M



train (base, frac=1):   5%|▌         | 1002/20000 [20:00<28:05:56,  5.32s/it, params=10.18M, train_loss=3.7315]


step   1000 | val_loss 7.4250 | params 10.18M



train (base, frac=1):   5%|▌         | 1004/20000 [20:00<19:45:08,  3.74s/it, params=10.18M, train_loss=2.9694]